In [1]:
from ingest import load_data
documents = load_data()

In [2]:
documents[0]

{'id': '2',
 'recipe_name': "Sarah's Homemade Applesauce",
 'total_time': '25 mins',
 'servings': '4',
 'ingredients': '4  apples - peeled, cored and chopped, ¾ cup water, ¼ cup white sugar, ½ teaspoon ground cinnamon',
 'directions': "Combine apples, water, sugar, and cinnamon in a saucepan; cover and cook over medium heat until apples are soft, about 15 to 20 minutes.\nAllow apple mixture to cool, then mash with a fork or potato masher until it is the consistency you like.\n\n\n\n\n\n\n\n\n\n\n\nPhoto by cookin'mama.\ncookin mama\n",
 'nutrition': 'Total Fat 0g 0%, Sodium 3mg 0%, Total Carbohydrate 32g 12%, Dietary Fiber 4g 13%, Total Sugars 27g, Protein 0g, Vitamin C 6mg 32%, Calcium 13mg 1%, Iron 0mg 1%, Potassium 150mg 3%'}

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [33]:
data_gen_instructions = """
Assume you are emulating a person looking for a quick recipe idea. 
Your task is to generate one query that a user might ask to get an observed recipe suggested. 
The query should sound like something a real user would type into a search engine or ask an AI assistant.

Each query should mention at least one of the following:
1. available cooking time,
2. ingredients or groceries the user has on hand,
3. dietary preference (e.g., vegetarian, vegan, gluten-free),
4. nutritional goal or eating preference (e.g., high protein, low carb, low fat, high fiber, lower calories).
The query should be consistent with the recipe.
If the query mentions available ingredients, they do not need to exactly match the recipe ingredients. Partial overlap is sufficient, as long as the recipe would still be a reasonable recommendation.
Nutritional goals should be expressed naturally, using qualitative preferences (e.g., "high protein", "something light", "lower carb") rather than exact nutrient values or calorie counts.
Use natural, conversational language that resembles real internet searches or AI assistant prompts.
""".strip()

In [34]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [35]:
doc = documents[300]

In [36]:
import json
user_prompt = json.dumps(doc)

In [37]:
doc

{'id': '550',
 'recipe_name': 'Persimmon Oatmeal Cookies',
 'total_time': '25 mins',
 'servings': '24',
 'ingredients': '1 ¾ cups rolled oats, 1 ½ cups sifted all-purpose flour, ½ cup English toffee-flavored baking bits (such as Heath®), 1 teaspoon baking soda, 1 teaspoon salt, ¼ teaspoon ground nutmeg, ½ teaspoon ground cinnamon, ¼ teaspoon ground cloves, 1 cup brown sugar, ¾ cup butter, 1  egg, 1 cup persimmon puree, 1 teaspoon vanilla',
 'directions': 'Preheat oven to 350 degrees F (175 degrees C). Line baking sheets with parchment paper.\nWhisk oats, flour, toffee bits, baking soda, salt, nutmeg, cinnamon, and cloves together in a bowl.\nBeat brown sugar and butter together in a bowl with an electric mixer until creamy, 2 minutes. Beat in egg. Beat persimmon puree and vanilla into butter mixture.\nMix flour mixture into persimmon mixture until dough is just moistened and no streaks of flour or oats remain. Drop spoonfuls of dough 1 1/2 inches apart onto ungreased baking sheets.\nBa

In [38]:
user_prompt

'{"id": "550", "recipe_name": "Persimmon Oatmeal Cookies", "total_time": "25 mins", "servings": "24", "ingredients": "1 \\u00be cups rolled oats, 1 \\u00bd cups sifted all-purpose flour, \\u00bd cup English toffee-flavored baking bits (such as Heath\\u00ae), 1 teaspoon baking soda, 1 teaspoon salt, \\u00bc teaspoon ground nutmeg, \\u00bd teaspoon ground cinnamon, \\u00bc teaspoon ground cloves, 1 cup brown sugar, \\u00be cup butter, 1  egg, 1 cup persimmon puree, 1 teaspoon vanilla", "directions": "Preheat oven to 350 degrees F (175 degrees C). Line baking sheets with parchment paper.\\nWhisk oats, flour, toffee bits, baking soda, salt, nutmeg, cinnamon, and cloves together in a bowl.\\nBeat brown sugar and butter together in a bowl with an electric mixer until creamy, 2 minutes. Beat in egg. Beat persimmon puree and vanilla into butter mixture.\\nMix flour mixture into persimmon mixture until dough is just moistened and no streaks of flour or oats remain. Drop spoonfuls of dough 1 1/2

In [39]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [40]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [41]:
response.output_parsed.questions

['Can you give me a quick 25-minute cookie recipe using persimmons and oats?']

In [42]:
from evaluation_utils import llm_structured

In [43]:
from evaluation_utils import llm_structured_retry

In [44]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [45]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [46]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/598 [00:00<?, ?it/s]

In [50]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

607

In [51]:
ground_truth[10]

{'question': 'What’s a quick 45-minute recipe for apple cinnamon muffins?',
 'document': '24'}

In [52]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.34313549999999976

In [53]:
import pandas as pd

In [54]:
df_ground_truth = pd.DataFrame(ground_truth)

In [55]:
df_ground_truth.to_csv("data/ground_truth_v3.csv", index=False)